# Train, diagnose, and evaluate the Khmer OCR recognizer

One notebook covering the full loop -- **train -> diagnose -> evaluate -> infer** --
on **Colab**, **Kaggle**, or a **local machine**, on **CPU, GPU, or TPU** (auto-detected,
no notebook changes needed).

Sections:
1. **Setup** -- clone/detect environment, install deps, fetch tokenizer.
2. **Data** -- pull samples from the 4 configured real_data sources, deduplicate.
3. **Train** -- runs `recognizer.train.run_training` with a small, fast-loop-friendly
   default config (not the 200k-step production default) so you can iterate quickly.
4. **Diagnose** -- CTC health check (blank-frac, whether the model has enough encoder
   frames per target length to even form a valid alignment) -- this is what caught a
   real bug during development (an LR-warmup schedule that never let the model reach
   peak LR, which kept CTC stuck predicting all-blank for an entire run).
5. **Evaluate** -- quantitative CER (AR-decoder + CTC-greedy) over the held-out val
   split, plus the train-loss / val-CER curves.
6. **Infer** -- run a handful of held-out images through the trained model and read
   ground-truth vs. CTC vs. AR predictions side by side.

**Notes learned the hard way (see recognizer/README.md):**
- `warmup_steps` must be well below `max_steps` (~10%) or the LR schedule never
  reaches peak and the model barely learns -- `train.py` now auto-clamps this with a
  loud warning if misconfigured, but set it deliberately for a short run.
- DataLoader workers (`num_workers > 0`) fork *after* CUDA is initialized in the
  notebook kernel, which is unsafe (can deadlock silently, GPU pinned at 100% with
  zero real steps). This notebook defaults to `num_workers=0` (synchronous loading)
  for that reason -- safe at the small/medium sample counts this notebook is meant
  for. Only raise it for a large from-a-script training run, not from a notebook cell.

**Before running:** optionally add an `HF_TOKEN` secret if you want checkpoints
pushed to the Hugging Face Hub (Colab: key icon in the left sidebar; Kaggle: Add-ons >
Secrets; local: `export HF_TOKEN=hf_...`). Not required to train/diagnose/evaluate
locally-only -- leave `PUSH_TO_HUB = False` below if you don't want this.

## 1. Setup

In [ ]:
import os, subprocess, sys

def detect_environment():
    if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ:
        return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/working"):
        return "kaggle"
    return "local"

ENV = detect_environment()
print("environment:", ENV)

REPO_URL = "https://github.com/Pich09/tuna-ocr.git"

if not os.path.isdir("recognizer"):
    subprocess.run(["git", "clone", REPO_URL, "tuna-ocr"], check=True)
    os.chdir("tuna-ocr")

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())


In [ ]:
!pip install -q -r recognizer/requirements.txt -r real_data/requirements.txt matplotlib editdistance


In [ ]:
from recognizer import env_utils

checkpoint_root = env_utils.get_checkpoint_root(ENV)
accelerator = env_utils.detect_accelerator()
print("checkpoint root:", checkpoint_root)
print("accelerator:", accelerator)
if accelerator == "cpu":
    print("no GPU/TPU detected -- training will be slow. On Colab: Runtime > Change "
          "runtime type. On Kaggle: Settings > Accelerator.")

PUSH_TO_HUB = False  # flip to True (and set HF_TOKEN as a platform secret) to push checkpoints
hf_token = env_utils.get_hf_token(ENV) if PUSH_TO_HUB else None


In [ ]:
# Downloads Panhapich/khmer-sp-8k's SentencePiece model + khmer_segmentation.py
# wrapper (a bare .model file is not enough -- see recognizer/README.md).
from recognizer.tokenizer.fetch_tokenizer import fetch_tokenizer

fetch_tokenizer()


## 2. Data

`NUM_SAMPLES_PER_SOURCE` controls how many rows are streamed per source. Keep this
small (hundreds to a couple thousand) for a diagnose/evaluate loop -- these are
notebook-scale runs, not a production training pull. `chanrith_ocr_image_line` alone
has 12M+ rows; do not raise this into the tens of thousands here without expecting a
very long pull.

In [ ]:
from pathlib import Path
from real_data.config import EXTERNAL_DATASETS, REAL_DATA_ROOT

NUM_SAMPLES_PER_SOURCE = 300  # small + fast; raise for a more serious run

real_data_roots = []
for source in EXTERNAL_DATASETS:
    out_dir = REAL_DATA_ROOT / "samples" / source
    real_data_roots.append(out_dir)
    if (out_dir / "manifest.tsv").exists():
        print(f"{source}: already present, skipping")
        continue
    print(f"{source}: pulling {NUM_SAMPLES_PER_SOURCE} samples...")
    !python -m real_data.generate_external_chunks --source {source} --num-samples {NUM_SAMPLES_PER_SOURCE}

print(real_data_roots)


In [ ]:
# Pool + deduplicate across sources (exact + near-duplicate image detection) before
# training ever sees the data. --near-dup-threshold 0 disables the O(n^2) near-dup
# pass -- only use 0 if you scale NUM_SAMPLES_PER_SOURCE into the tens of thousands+,
# where the default pairwise comparison would be too slow.
dedup_manifest = REAL_DATA_ROOT / "samples" / "dedup" / "manifest.tsv"
real_data_dirs_str = " ".join(str(p) for p in real_data_roots)
!python -m real_data.deduplicate --real-data-dirs {real_data_dirs_str} --out-dir {dedup_manifest.parent}


## 3. Train

Defaults here are tuned for a **short, fast diagnose/evaluate loop**, not a full
production run -- see `recognizer/README.md` / `notebooks/train_recognizer.ipynb` for
a long production-scale config instead.

**`warmup_steps` must scale with `max_steps`** (rule of thumb: ~10%). `train.py`
auto-clamps and warns if you get this wrong, but for a short run like this, set it
deliberately -- e.g. `warmup_steps=300` for `max_steps=3000`. A real training bug in
this project's history was exactly this: `warmup_steps=4000` left over from the
long-run default, paired with an overridden `max_steps=3000`, meant the run never
reached peak LR and CTC stayed stuck predicting blank for the entire run.

In [ ]:
from recognizer.config import ModelConfig, TrainConfig
from recognizer.train import run_training

RUN_NAME = "notebook_run"
MAX_STEPS = 3000
WARMUP_STEPS = 300   # ~10% of MAX_STEPS -- see note above
EVAL_EVERY = 300     # quantitative val CER (AR + CTC), logged to eval_log.csv
SAMPLE_EVERY = 300   # qualitative gt/ctc/ar prediction dump, logged to train.log
CKPT_EVERY = 300

model_cfg = ModelConfig()
train_cfg = TrainConfig(
    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    eval_every=EVAL_EVERY,
    sample_every=SAMPLE_EVERY,
    ckpt_every=CKPT_EVERY,
    log_every=100,
    num_workers=0,  # see the "notes learned the hard way" cell at the top -- fork+CUDA
                    # in a notebook kernel can deadlock; 0 is the safe default here.
)

model = run_training(
    model_cfg, train_cfg,
    dedup_manifest_path=dedup_manifest,
    checkpoint_root=checkpoint_root,
    run_name=RUN_NAME,
    push_to_hub=PUSH_TO_HUB,
    repo_id="Panhapich/tuna-ocr",
    hf_token=hf_token,
    hub_private=True,
    auto_batch_size=True,  # OOM-probing auto-tune; no-op on CPU/TPU
)

run_dir = Path(checkpoint_root) / RUN_NAME
print("run dir:", run_dir)


## 4. Diagnose

Two health checks that catch real, silent failure modes early rather than after a
long run:

- **CTC feasibility**: CTC needs encoder frames `T` >= target length `L` for a valid
  alignment to even exist; a sample where `T < L` contributes zero gradient
  (`zero_infinity=True` silently zeroes it). If many samples fail this, CTC can't
  learn regardless of LR/steps -- usually means `chunk_width` is too small for these
  transcripts, or a source has unusually long lines.
- **Blank collapse**: CTC always starts by predicting all-blank (the easiest low-loss
  path early on); it should break out of this within the first few hundred steps once
  LR is near its peak. If `blank_frac` is pinned at 1.0 for a long time despite LR
  being near peak, something is wrong (as opposed to just being early in training) --
  check the LR schedule first (see the warmup note in section 3).

In [ ]:
import random
import torch
import torch.nn.functional as F

from recognizer.data.manifest import load_dedup_manifest
from recognizer.data.transforms import chunk_line_image
from recognizer.evaluate import load_model

def diagnose_ctc(checkpoint_path, manifest_path=dedup_manifest, val_frac=0.02, seed=0, device=None):
    """Runs the CTC feasibility + blank-collapse checks over the same
    deterministic held-out val split train.py uses internally."""
    from recognizer.tokenizer.khmer_ocr_tokenizer import KhmerOcrTokenizer
    from recognizer.config import TOKENIZER_ASSETS_DIR

    device = device or torch.device("cpu")  # CPU: don't contend with a live training run's GPU
    tokenizer = KhmerOcrTokenizer(TOKENIZER_ASSETS_DIR)
    model, cfg, char_vocab = load_model(checkpoint_path, tokenizer, device)
    blank_id = char_vocab.blank_id

    samples = list(load_dedup_manifest(manifest_path))
    random.Random(seed).shuffle(samples)
    n_val = max(1, int(len(samples) * val_frac))
    val = samples[:n_val]

    n_ge, blank_fracs = 0, []
    with torch.no_grad():
        for s in val:
            chunks, _ = chunk_line_image(s.image_path, cfg.chunk_width, cfg.chunk_overlap, cfg.img_height)
            cb = torch.stack(chunks)
            cpl = torch.tensor([len(chunks)], dtype=torch.long)
            enc_out, enc_lengths, _ = model.encode(cb, cpl)
            T = int(enc_lengths[0])
            L = len(char_vocab.encode(s.text))
            logits = model.ctc_head(enc_out)[0][:T]
            blank_fracs.append((logits.argmax(-1) == blank_id).float().mean().item())
            n_ge += (T >= L)

    print(f"CTC feasibility: {n_ge}/{len(val)} val samples have T >= L "
          f"({100*n_ge/len(val):.0f}% -- these are the only samples CTC can learn from)")
    print(f"blank_frac over val: mean={sum(blank_fracs)/len(blank_fracs):.3f}  "
          f"(near 1.0 = still in/stuck-in blank collapse; well under 1.0 = learning to spike characters)")
    return n_ge / len(val), sum(blank_fracs) / len(blank_fracs)

# Point this at whichever checkpoint you want to inspect -- defaults to the run above's latest.
latest_ckpt = run_dir / "last.pt"
diagnose_ctc(latest_ckpt)


## 5. Evaluate

Quantitative Character Error Rate (CER) over the exact same held-out val split
`train.py` used internally (deterministic shuffle + `val_frac` head slice) -- so this
never accidentally scores the model on samples it trained on. Reports both the
AR-decoder's CER and the CTC-greedy CER (an independent, alignment-free sanity
check) -- a large gap between the two, or CTC CER stuck near 1.0, is itself a useful
diagnostic (pairs with section 4).

In [ ]:
import csv

from recognizer.data.dataset import OCRLineDataset, make_collate_fn
from recognizer.evaluate import compute_cer, ctc_greedy_decode
from torch.utils.data import DataLoader

def evaluate_val_split(checkpoint_path, manifest_path=dedup_manifest, val_frac=0.02, seed=0, device=None):
    from recognizer.tokenizer.khmer_ocr_tokenizer import KhmerOcrTokenizer
    from recognizer.config import TOKENIZER_ASSETS_DIR

    device = device or torch.device("cpu")
    tokenizer = KhmerOcrTokenizer(TOKENIZER_ASSETS_DIR)
    model, cfg, char_vocab = load_model(checkpoint_path, tokenizer, device)

    samples = list(load_dedup_manifest(manifest_path))
    random.Random(seed).shuffle(samples)
    n_val = max(1, int(len(samples) * val_frac))
    val_samples = samples[:n_val]

    val_ds = OCRLineDataset(val_samples, tokenizer, cfg, char_vocab=char_vocab)
    collate_fn = make_collate_fn(tokenizer, cfg.chunk_width)
    loader = DataLoader(val_ds, batch_size=8, shuffle=False, collate_fn=collate_fn)

    refs, ar_hyps, ctc_hyps = [], [], []
    with torch.no_grad():
        for batch in loader:
            enc_out, enc_lengths, enc_block_ids = model.encode(batch["chunks"], batch["chunks_per_line"])
            max_blocks = int(batch["chunks_per_line"].max())
            ar_tokens = model.decoder.decode_greedy(enc_out, enc_lengths, enc_block_ids, max_blocks)
            ctc_logits = model.ctc_head(enc_out)
            ctc_tokens = ctc_greedy_decode(ctc_logits, char_vocab.blank_id)
            for text, ar_row, ctc_row in zip(batch["texts"], ar_tokens.tolist(), ctc_tokens):
                refs.append(text)
                ar_hyps.append(tokenizer.decode(ar_row))
                ctc_hyps.append(char_vocab.decode(ctc_row))

    ar_cer = compute_cer(refs, ar_hyps)
    ctc_cer = compute_cer(refs, ctc_hyps)
    print(f"AR-decoder val CER: {ar_cer:.4f}  |  CTC-greedy val CER: {ctc_cer:.4f}  (n={len(refs)})")
    return ar_cer, ctc_cer

evaluate_val_split(latest_ckpt)


### Loss / CER curves

Reads `train_log.csv` (per-`log_every`-step train loss) and `eval_log.csv`
(per-`eval_every`-step val CER) straight from the run directory -- no re-running
needed, these were written live during training.

In [ ]:
import matplotlib.pyplot as plt

def plot_curves(run_dir):
    run_dir = Path(run_dir)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    loss_path = run_dir / "train_log.csv"
    if loss_path.exists():
        with loss_path.open() as f:
            rows = list(csv.DictReader(f))
        steps = [int(r["step"]) for r in rows]
        axes[0].plot(steps, [float(r["loss"]) for r in rows], label="loss")
        axes[0].plot(steps, [float(r["ctc_loss"]) for r in rows], label="ctc_loss")
        axes[0].plot(steps, [float(r["ce_loss"]) for r in rows], label="ce_loss")
        axes[0].set_xlabel("step"); axes[0].set_title("train loss"); axes[0].legend()
    else:
        axes[0].set_title("train_log.csv not found yet")

    eval_path = run_dir / "eval_log.csv"
    if eval_path.exists():
        with eval_path.open() as f:
            rows = list(csv.DictReader(f))
        if rows:
            steps = [int(r["step"]) for r in rows]
            axes[1].plot(steps, [float(r["val_ar_cer"]) for r in rows], label="val_ar_cer", marker="o")
            axes[1].plot(steps, [float(r["val_ctc_cer"]) for r in rows], label="val_ctc_cer", marker="o")
            axes[1].axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
            axes[1].set_xlabel("step"); axes[1].set_title("held-out val CER (lower is better)"); axes[1].legend()
        else:
            axes[1].set_title("eval_log.csv has no rows yet")
    else:
        axes[1].set_title("eval_log.csv not found (eval_every=0?)")

    plt.tight_layout(); plt.show()

plot_curves(run_dir)


## 6. Infer

Runs the trained model on a handful of held-out images and shows the image alongside
ground truth vs. CTC-greedy vs. AR-decoder predictions -- the most direct "is this
actually reading" check, complementing the CER number above.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def show_predictions(checkpoint_path, manifest_path=dedup_manifest, n=5, val_frac=0.02, seed=0, device=None):
    from recognizer.tokenizer.khmer_ocr_tokenizer import KhmerOcrTokenizer
    from recognizer.config import TOKENIZER_ASSETS_DIR

    device = device or torch.device("cpu")
    tokenizer = KhmerOcrTokenizer(TOKENIZER_ASSETS_DIR)
    model, cfg, char_vocab = load_model(checkpoint_path, tokenizer, device)

    samples = list(load_dedup_manifest(manifest_path))
    random.Random(seed).shuffle(samples)
    n_val = max(1, int(len(samples) * val_frac))
    val_samples = samples[:n_val][:n]

    fig, axes = plt.subplots(len(val_samples), 1, figsize=(8, 2 * len(val_samples)))
    if len(val_samples) == 1:
        axes = [axes]

    with torch.no_grad():
        for ax, s in zip(axes, val_samples):
            chunks, _ = chunk_line_image(s.image_path, cfg.chunk_width, cfg.chunk_overlap, cfg.img_height)
            cb = torch.stack(chunks)
            cpl = torch.tensor([len(chunks)], dtype=torch.long)
            enc_out, enc_lengths, enc_block_ids = model.encode(cb, cpl)
            max_blocks = min(len(chunks), 256 // max(1, cfg.max_tokens_per_block) + 1)
            ar_tokens = model.decoder.decode_greedy(enc_out, enc_lengths, enc_block_ids, max_blocks)[0].tolist()
            ar_txt = tokenizer.decode(ar_tokens, strip_control=True)
            ctc_logits = model.ctc_head(enc_out)[:, :int(enc_lengths[0])]
            ctc_txt = char_vocab.decode(ctc_greedy_decode(ctc_logits, char_vocab.blank_id)[0])

            ax.imshow(Image.open(s.image_path).convert("L"), cmap="gray")
            ax.axis("off")
            ax.set_title(f"gt: {s.text}\nctc: {ctc_txt}\nar: {ar_txt}", fontsize=9, loc="left")

    plt.tight_layout(); plt.show()

show_predictions(latest_ckpt, n=5)
